# Figures 1 & S2 — Whole-brain c-Fos response to morphine and withdrawal

Generates main-text panels **1B, 1C, 1D** and supplement **S2** from the main whole-brain c-Fos dataset
(43 animals, 5–6 drug conditions).

**Data:** download the Figshare deposit and point `OPIOID_DATA_ROOT` at it (see the
path cell below). All inputs resolve from there via the Figshare folder layout.
Associated supplements (S2 raw-count maps, S3 lateralization) are in separate notebooks.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tifffile
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import dask.array as da

# brainvis package (install from GitHub; see requirements.txt):
#   pip install git+https://github.com/kenjp1223/brainvis
from brain_vis import overlap_contour, set_transparency, get_subregions

## Paths

`DATA_ROOT` points at the downloaded **Figshare deposit**, whose folder layout mirrors
this repo's data-source groups. Set the `OPIOID_DATA_ROOT` environment variable, or edit
the default below. Rendered panels are written to the repo (`../figures`).

In [ ]:
# >>> Set this to the downloaded Figshare deposit (or export OPIOID_DATA_ROOT) <<<
DATA_ROOT = Path(os.environ.get("OPIOID_DATA_ROOT", "/path/to/Figshare_deposit")).expanduser()

GROUP = DATA_ROOT / "01_main_cfos_morphine"   # this dataset (raw + processed)
ATLAS = DATA_ROOT / "shared" / "atlas"        # shared atlas / ontology

FIG_OUT = Path("../figures")                  # rendered panels -> repo
FIG_OUT.mkdir(parents=True, exist_ok=True)

figure_key = "Figure1"
assert GROUP.exists(), f"DATA_ROOT not found: {DATA_ROOT}. Set OPIOID_DATA_ROOT."

In [ ]:
# Plot style: vector text editable in Illustrator, transparent backgrounds
plt.rcParams.update({
    'figure.facecolor': 'none',
    'axes.facecolor': 'none',
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'legend.facecolor': 'none',
    'legend.edgecolor': 'none',
    'text.color': 'black',
    'font.family': 'Arial',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

## Load metadata, atlas, and ontology

In [ ]:
# Animal metadata (drop A7 = missing data)
metadf = pd.read_csv(GROUP / "OP_meta.csv", index_col=False)
metadf = metadf[metadf.ID != 'A7'].reset_index(drop=True)

# Unified (Kim) mouse-brain atlas: ontology table, contour map, label volume
atlas_df = pd.read_csv(
    ATLAS / "atlas_info_KimRef_FPbasedLabel_v4.0_brain_with_size_with_curated_with_cleaned_acronyms.csv",
    index_col=False)
contour_img = tifffile.imread(ATLAS / "Kim_ref_adult_FP-label_v2.9_contour_map.tif")
atlas_img   = tifffile.imread(ATLAS / "Kim_ref_adult_FP-label_v4.0.tif")

# Curated region lists
with open(ATLAS / "curated_acronym.pickle", "rb") as handle:
    curated_acronyms = pickle.load(handle)
with open(ATLAS / "ancestor_curated_acronym.pickle", "rb") as handle:
    ancestor_curated_acronyms = pickle.load(handle)

# files flagged as fully processed
fnames = [f for f in metadf.fname.values if 'DONE' in f]

In [ ]:
# Drug conditions (morphine-related groups) and display labels/colors
Conditions = ['Saline', 'Acute_Morphine', 'Chronic_Morphine',
              'Withdrawal_Morphine', 'Chronic_Morphine_21', 'Withdrawal_Morphine_21']
Condition_figure_name = ['Saline', 'Acute', 'Chronic', 'Early WD', 'Re-exposure', 'Late WD']
Condition_color = ['gray', 'lime', 'orange', 'cyan', 'blue', 'purple']

metadf = metadf[metadf.Condition.isin(Conditions)]

In [ ]:
# Region-level results (long) and per-animal region-density heatmap (wide)
pivot_heatmap_df = pd.read_csv(GROUP / "OP_cFos_heatmap.csv", index_col=0)
pivot_heatmap_df = pivot_heatmap_df[metadf[metadf.Condition.isin(Conditions)]['ID'].values]

merge_df = pd.read_csv(GROUP / "OP_cFos_full_result.csv", index_col=0)
merge_df = merge_df[merge_df.Condition.isin(Conditions)]
merge_df = merge_df[merge_df.fname.isin(fnames)]

# sanity checks
assert (merge_df.fname.unique() == fnames).all()
assert (pivot_heatmap_df.columns == metadf.ID.unique()).all()
print("Animals per group:")
print(metadf.groupby('Condition').size())

Remove the hindbrain (HB) and cerebellum (CBL) subtrees: poor registration quality and
outside the scope of this analysis.

In [ ]:
# ancestor regions used to organize the ontology (HB/CBL excluded)
unique_ancestor_curated_acronyms = ['Isocortex', 'OLF', 'HPF', 'CTXsp', 'STR', 'PAL', 'TH', 'HY', 'MB']
ancestor_names = [atlas_df.loc[atlas_df.acronym == f, 'name'].values[0] for f in unique_ancestor_curated_acronyms]
ancestor_idxs  = [atlas_df.loc[atlas_df.acronym == f, 'id'].values[0]   for f in unique_ancestor_curated_acronyms]

# drop HB + CBL subtrees from the region table
remove_ancestor_ids = atlas_df[(atlas_df.acronym == 'HB') | (atlas_df.acronym == 'CBL')]['id'].values
remove_df = pd.concat(
    [get_subregions(atlas_df, idx, return_original=True) for idx in remove_ancestor_ids],
    axis=0)
sub_atlas_df = atlas_df.set_index(['id']).drop(remove_df['id'].values)
merge_df = merge_df[merge_df.acronym.isin(sub_atlas_df.acronym.unique())]

## Build the design matrix for the GLM

Experiment variables = 5 drug conditions (saline as baseline); covariates = body weight,
age, sex. Staining batch could not be included because some conditions coincide with batch.

In [ ]:
# age in days; per-region atlas metadata
metadf['age'] = [(datetime.strptime(pday, '%m/%d/%Y') - datetime.strptime(dob, '%m/%d/%Y')).days
                 for pday, dob in metadf.loc[:, ['Date_Perfusion', 'DOB']].values]
atlasmeta = merge_df.reset_index().loc[
    merge_df.reset_index().ID == 'A1', ['id', 'parent_id', 'acronym', 'name', 'parent_acronym']]

In [ ]:
# categorical encodings -> dummy columns
sex_category       = pd.CategoricalDtype(categories=['F', 'M'], ordered=False)
condition_category = pd.CategoricalDtype(categories=Conditions, ordered=True)

merge_df['Sex']       = merge_df['Sex'].astype(sex_category)
merge_df['Condition'] = merge_df['Condition'].astype(condition_category)

condition_dummies = pd.get_dummies(merge_df['Condition'])
sex_dummies       = pd.get_dummies(merge_df['Sex']).loc[:, ['F']].rename(columns={'F': 'Sex_d'})  # female = 1

merge_df = pd.concat([merge_df, condition_dummies, sex_dummies], axis=1)

## Figure 1B — Whole-brain c-Fos+ cell counts per condition

In [ ]:
pannel_key = 'B'

fig, axs = plt.subplots(1, 1, figsize=(1, 2))
merge_df.Condition = merge_df.Condition.astype('str')
tdata = (merge_df.loc[merge_df.parent_acronym == 'grey', ['ID', 'Condition', 'newcounts']]
         .groupby(['ID', 'Condition']).sum().reset_index().dropna())

sns.stripplot(data=tdata, y='Condition', x='newcounts',
              order=Conditions, ax=axs, palette=Condition_color, alpha=0.25)
sns.pointplot(data=tdata, y='Condition', x='newcounts',
              order=Conditions, ax=axs, palette=Condition_color,
              markers="o", markersize=4, linestyle="none", linewidth=0.5)
sns.despine()
axs.set_xlabel('# of whole brain\nc-Fos+ cells')
axs.set_yticklabels(Condition_figure_name, rotation=0)
axs.set_xlim(0,)

fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}.png', bbox_inches='tight', dpi=216)
fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}.pdf', bbox_inches='tight', dpi=216)

## Linear regression (predictive activation maps)

Voxel-wise OLS of the c-Fos+ cell-count map on experiment + non-experiment variables.
The condition beta coefficients (relative to saline) are the condition-specific
**predictive activation scores** visualized in 1C–1D. The spatial heatmap is stored as a
zarr array; row order follows `OP_cFos_fnamelist.npy`.

In [ ]:
# variables, one row per animal (region 'CH' = whole grey-matter root)
variable_df = merge_df.loc[merge_df.acronym == 'CH', ['fname', 'Condition', 'BW', 'Age', 'Sex_d'] + Conditions]
variable_df = variable_df.set_index('fname')

# order rows to match the spatial heatmap
fnamelist = np.load(GROUP / "OP_cFos_fnamelist.npy")
variable_df = variable_df.loc[fnamelist, :]
metadf = metadf.set_index('fname').loc[fnamelist, :]

In [ ]:
# voxels that belong to included brain regions (exclude background / removed subtrees)
brain_voxels = np.where(np.isin(atlas_img.flatten(), atlas_df['id'].values))[0]

In [ ]:
from sklearn.preprocessing import StandardScaler

# standardize continuous covariates
scaler = StandardScaler(with_mean=True)
variable_df[['BW', 'Age']] = scaler.fit_transform(variable_df[['BW', 'Age']])

# saline = baseline -> drop; add intercept; order rows to metadf
variable_df = variable_df.drop(columns='Saline')
variable_df['constant'] = 1
variable_df = variable_df.loc[metadf.index]
variable_df = variable_df.drop(columns='Condition')
variables = np.array(variable_df.astype('float'))

In [ ]:
# load the spatial c-Fos heatmap (zarr) and keep only brain voxels
heatmap_da = da.from_zarr(str(GROUP / "OP_cFos_heatmap_array"), mode="r")
brain_heatmap = heatmap_da[:, brain_voxels].compute()

In [ ]:
import statsmodels.api as sm

# voxel-wise OLS: (n_animals x n_voxels) on (n_animals x n_variables)
models = sm.OLS(brain_heatmap, variables).fit()

In [ ]:
# write one beta map per variable to the Figshare data folder (intermediate outputs)
for variable_idx, variable_name in enumerate(variable_df.columns):
    temp_img = np.zeros_like(atlas_img.flatten(), dtype='float64')
    temp_img[brain_voxels] = models.params[variable_idx, :]
    np.save(GROUP / f'{variable_name}_betas.npy', np.reshape(temp_img, atlas_img.shape))
print("Saved beta maps for:", list(variable_df.columns))

## Figure 1C — Predictive activation maps, single coronal plane

Beta-coefficient maps (cmin/cmax = ±15) overlaid on the atlas contour, one panel per
variable at a representative z-plane.

In [ ]:
pannel_key = 'C'

imy_slice = slice(50, 400)
imx_slice = slice(75, 575)
zplace = 125

fig, axs = plt.subplots(2, int(np.ceil(len(variable_df.columns) / 2)),
                        figsize=(20, np.ceil(len(variable_df.columns) / 2)))
for cidx, variable_name in enumerate(variable_df.columns):
    ax = axs.flatten()[cidx]
    beta_array = np.load(GROUP / f'{variable_name}_betas.npy')
    base_image_color, overlayed_image = overlap_contour(
        beta_array, contour_img, cmin=-15, cmax=15, outputpath=False)
    trans_img = set_transparency(overlayed_image[zplace, :, :], (atlas_img == 0)[zplace, :, :])
    ax.imshow(trans_img[imy_slice, imx_slice])
    ax.axis('off')
    ax.set_title(f'{variable_name}_z={zplace}')

fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}.png', dpi=216, bbox_inches='tight')
fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}.pdf', dpi=216, bbox_inches='tight')

## Figure 1D — Predictive activation maps across coronal planes

Per drug condition (relative to saline), beta maps along the A–P axis at curated
z-planes. Saline (baseline, no beta) is shown as the mean raw density for reference.

In [ ]:
pannel_key = 'D'

curated_zplanes = [84, 104, 117, 153, 186, 220]
imy_slice = slice(25, 425)
imx_slice = slice(50, 600)

# condition beta maps (+ intercept)
for condition in Conditions[1:] + ['constant']:
    theatmap = np.load(GROUP / f'{condition}_betas.npy')
    __, overlayed_image = overlap_contour(theatmap, contour_img, cmin=-15, cmax=15, outputpath=None)

    fig, axs = plt.subplots(1, len(curated_zplanes), figsize=(3 * len(curated_zplanes), 3), sharey=True)
    fig.subplots_adjust(wspace=0.25, hspace=0.3)
    for idx, ax in enumerate(axs):
        trans_img = set_transparency(overlayed_image[curated_zplanes[idx], :, :],
                                     (atlas_img == 0)[curated_zplanes[idx], :, :])
        ax.imshow(trans_img[imy_slice, imx_slice])
        ax.axis('off')
        ax.set_ylabel(condition, color='black')
    fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}_{condition}_betacoef.png', bbox_inches='tight', dpi=1024)
    fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}_{condition}_betacoef.pdf', bbox_inches='tight', dpi=1024)

In [ ]:
# Saline reference: mean raw density (viridis), same z-planes
condition = 'Saline'
theatmap = np.mean(heatmap_da[(metadf.Condition == 'Saline'), :], axis=0).reshape(atlas_img.shape)

fig, axs = plt.subplots(1, len(curated_zplanes), figsize=(3 * len(curated_zplanes), 3), sharey=True)
fig.subplots_adjust(wspace=0.25, hspace=0.3)
for idx, ax in enumerate(axs):
    __, overlayed_image = overlap_contour(theatmap, contour_img, cmin=0, cmax=10,
                                          outputpath=None, colormap=plt.cm.viridis)
    trans_img = set_transparency(overlayed_image[curated_zplanes[idx], :, :],
                                 (atlas_img == 0)[curated_zplanes[idx], :, :])
    ax.imshow(trans_img[imy_slice, imx_slice])
    ax.axis('off')
    ax.set_ylabel(condition, color='black')
fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}_{condition}_mean.png', bbox_inches='tight', dpi=1024)
fig.savefig(FIG_OUT / f'{figure_key}{pannel_key}_{condition}_mean.pdf', bbox_inches='tight', dpi=1024)

## Figure S2 — Brain-wide c-Fos+ cell density across conditions (related to Figure 1)

Representative 50-µm coronal planes of mean c-Fos+ cell density (per mm³) for each
condition (hot colormap, 0–15), at the same curated z-planes as 1D.

In [ ]:
supp_key = 'FigureS2'

imy_slice = slice(25, 425)
imx_slice = slice(50, 600)

for condition in Conditions:
    theatmap = np.nanmean(heatmap_da[metadf.Condition == condition], axis=0).reshape(atlas_img.shape).compute()
    __, overlayed_image = overlap_contour(theatmap, contour_img, cmin=0, cmax=15,
                                          outputpath=None, colormap=plt.cm.hot, overlap_black=False)

    fig, axs = plt.subplots(1, len(curated_zplanes), figsize=(3 * len(curated_zplanes), 3), sharey=True)
    fig.subplots_adjust(wspace=0.25, hspace=0.3)
    for idx, ax in enumerate(axs):
        trans_img = set_transparency(overlayed_image[curated_zplanes[idx], :, :],
                                     (atlas_img == 0)[curated_zplanes[idx], :, :])
        ax.imshow(trans_img[imy_slice, imx_slice])
        ax.axis('off')
        ax.set_ylabel(condition, color='black')
    fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin=0, vmax=15), cmap=plt.cm.hot),
                 ax=axs, fraction=0.03, pad=0.02)
    fig.savefig(FIG_OUT / f'{supp_key}_{condition}_rawcount.png', bbox_inches='tight', dpi=1024)
    fig.savefig(FIG_OUT / f'{supp_key}_{condition}_rawcount.pdf', bbox_inches='tight', dpi=1024)